In [1]:
%pip install duckdb
print("duckdb is ready")


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
duckdb is ready


In [2]:
import duckdb

print("⏳ 正在啟動 DuckDB 執行 Transaction Banking 數據聚合...")


sql_query = """
SELECT 
    type AS transaction_channel,
    COUNT(*) AS total_transactions,
    SUM(ABS(net_amount)) AS total_cash_flow_volume,
    
    -- 統計各渠道觸發流動性預警的筆數
    SUM(CASE WHEN liquidity_risk_alert = True THEN 1 ELSE 0 END) AS high_risk_liquidity_events,
    
    -- 計算平均賬戶餘額 (了解客戶 Cash Buffer)
    ROUND(AVG(current_balance), 2) AS avg_client_balance,
    
    -- RM 潛在可 Cross-sell 短期融資產品的預估客戶交易量
    SUM(CASE WHEN liquidity_risk_alert = True THEN ABS(net_amount) ELSE 0 END) AS potential_credit_facility_opportunity

FROM 'clean_tb_transactions.csv'
GROUP BY type
ORDER BY high_risk_liquidity_events DESC;
"""


result = duckdb.sql(sql_query)

print("\n--- 📊 DuckDB SQL 分析結果（Transaction Banking 摘要）---")
print(result)

# 3. 匯出 CSV 檔供 Tableau 做 Dashboard
result.write_csv("tb_cash_management_summary.csv")
print("\n✅ SQL 分析完成！已匯出為 tb_cash_management_summary.csv")

⏳ 正在啟動 DuckDB 執行 Transaction Banking 數據聚合...

--- 📊 DuckDB SQL 分析結果（Transaction Banking 摘要）---
┌─────────────────────┬────────────────────┬────────────────────────┬────────────────────────────┬────────────────────┬───────────────────────────────────────┐
│ transaction_channel │ total_transactions │ total_cash_flow_volume │ high_risk_liquidity_events │ avg_client_balance │ potential_credit_facility_opportunity │
│       varchar       │       int64        │         double         │           int128           │       double       │                double                 │
├─────────────────────┼────────────────────┼────────────────────────┼────────────────────────────┼────────────────────┼───────────────────────────────────────┤
│ CASH_OUT            │            2237500 │     394412995224.49084 │                    2085170 │           17474.19 │                    378295893165.60236 │
│ PAYMENT             │            2151495 │      28093371138.36996 │                    1615576 │       